In [ ]:
import dash
import dash_core_components as dcc
import dash_html_components as html
import pandas as pd
import numpy as np
import plotly_express as px
from dash.dependencies import Input, Output
rootfold = '/home/j0hndoe/Documents/git/reddit-disinformation'

In [ ]:
def get_flag_perc(df, groups):
    return df.\
        groupby(groups)\
            [['n_subm_kept','ISFLAG']].\
                agg(sum).sort_index().reset_index().\
        assign(perc_FLAG = lambda df: df.ISFLAG/df.n_subm_kept)

In [ ]:
df_flags = pd.read_csv(rootfold+"/dashboard/data/app_flags_large.csv")
df_day = pd.read_csv(rootfold+"/dashboard/data/app_subr_day.csv")

df_day['week'] = df_day['week'].astype('int')
min_week, max_week = df_day['week'].min(), df_day['week'].max()
weekrange = [min_week, max_week]

filtered_df_day = df_day[(df_day['week'].between(*weekrange))]# & (df['varr']==varr)]
filtered_df_day['ISFLAG'] = filtered_df_day.flag.copy()#loc[:,flagtypes].sum(axis=1)>0
### Aggregate
filtered_df_week = get_flag_perc(
    filtered_df_day,
    groups = ['subreddit_cat','subreddit','week'])
filtered_df_agg = get_flag_perc(
    filtered_df_week,
    groups = ['subreddit_cat','subreddit'])

In [ ]:
df_flags

In [ ]:
filtered_df_week_cat = get_flag_perc(filtered_df_week, ['subreddit_cat','week'])
subr_line_all = px.line(
                    filtered_df_week_cat,
                    x = "week",
                    y = "ISFLAG",
                    color = "subreddit_cat")
subr_line_all.show()

In [ ]:
fig_subr_treemap = px.treemap(filtered_df_agg, 
                      path=['subreddit_cat', 'subreddit'], 
                      values='ISFLAG',
                      color='subreddit_cat')
fig_subr_treemap.show()